In [ ]:
import os

import json
import pickle
import networkx as nx
import pandas as pd
import numpy as np

import networkx as nx

import matplotlib.pyplot as plt
from utils.functions import load_yaml
from utils.graphfunction import load_graph, get_sample, get_uniprot_from_nodes, get_pos_from_nodes, get_res_from_nodes, get_node_id_rm_copy

In [ ]:
config = load_yaml("../config/RRGCL.yaml")
DATABASE = config.DATABASE

In [ ]:
finalG = load_graph((f"{DATABASE}/cleaned_weighted_graph.pkl"))
gnomad_mut_df = pd.read_csv((f"{DATABASE}/gnomad_mutation_counts_freq_final.csv"))
mut_df = pd.read_csv((f"{DATABASE}/node_mutation_occurrence_essential.csv"))
# ss_df = pd.read_csv((f"{DATABASE}/merged_ss_sctrcutre.csv"))

y_df = pd.read_csv("../node_features_with_location_nodeid_v031026.csv")

node_in_G = list(finalG.nodes())

# Graph EDA

In [ ]:
cc_list = list(nx.connected_components(finalG))
len(cc_list)

In [ ]:
n_in_cc = []
for cc_ in cc_list:
    n_in_cc.append(len(cc_))
n_in_cc.sort(reverse=True)

print("Top5 Size", n_in_cc[:5])

In [ ]:
plt.figure(figsize=(7,3))
plt.hist(n_in_cc[5:], color='skyblue', edgecolor='black', bins=100)
plt.yscale('log')
plt.show()

# Filtering Non-reviewed Protein

In [ ]:
# rm_uniprot = [
#     "a0a1x8xl64",
#     "b4dr52",
#     "b2r5b3"
# ]

In [ ]:
# proc_edge_df = pd.read_csv(f"{DATABASE}/step5_rmStrangeEdge_exceptCDSFilter_human_aa_edges_exclubq_Nucleosome_related_data_v031626.csv")
# proc_edge_df.head(3)
# len(proc_edge_df)

In [ ]:
# rm_nonReview_df = proc_edge_df[
#     (~proc_edge_df['remove_homo_uniprot1'].isin(rm_uniprot)) & 
#     (~proc_edge_df['remove_homo_uniprot2'].isin(rm_uniprot))
# ]
# len(rm_nonReview_df)

In [ ]:
# from generation.graph.helper import GenerateGraph_from_edge
# newG = GenerateGraph_from_edge(rm_nonReview_df, src='nodeid_1', tar='nodeid_2', weight_col='cleaned_total_energy')
# print(newG)

In [ ]:
# with open(os.path.join(DATABASE, 'cleaned_weighted_graph.pkl'), 'wb') as f:
#     pickle.dump(newG, f)

In [ ]:
# g = load_graph(os.path.join(DATABASE, 'cleaned_weighted_graph.pkl'))

In [ ]:
# print(g)

# Mutation Profile

## Overall Analysis

## AAindex

### AAindex1

### AAindex2

## Alpha Missense

In [ ]:
am_aa = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']

In [ ]:
with open(f"{DATABASE}/am_features.json", 'rb') as f:
    am = json.load(f)

In [ ]:
gnomad_df[gnomad_df['total_mutations_count']>10000]

## Gnomad Mutation

### Manual ADD Mutation Information

In [ ]:
# from generation.mutation.gnomad.helper import process_gnomad_csv
# add_files = ['O60814_H2BC12_gnomAD_v4.1.1_ENST00000356950.csv',
#              'P11388_TOP2A_gnomAD_v4.1.1_ENST00000423485.csv',
#              'Q5EE01_CENPW_gnomAD_v4.1.1_ENST00000368328.csv',
#              'Q96KQ7_EHMT2_gnomAD_v4.1.1_ENST00000375537.csv'
#              ]

In [ ]:
# c_gnomad_mut_df = gnomad_mut_df.copy()
# for file in add_files:
#     print(file)
#     uniprot = file.split("_")[0]
#     ENST = file.split("_")[-1].split(".")[0]
#     gene_symbol= file.split("_")[1]
#     df_raw = pd.read_csv(f"reference/{file}")
#     proc_df = process_gnomad_csv(df_raw, uniprot, ENST, gene_symbol=gene_symbol)
#     c_gnomad_mut_df = pd.concat([c_gnomad_mut_df, proc_df], axis=0)
# c_gnomad_mut_df.to_csv('proc_data/gnomad_mutation_counts_freq_final.csv', index=False)
# c_gnomad_mut_df.tail(5)

### EDA

In [ ]:
gnomad_mut_df.head(4)

In [ ]:
covered_nodes = []
node_in_gnomad = gnomad_mut_df['node_id'].tolist()

for n in node_in_G:
    cleaned_node_id = get_node_id_rm_copy(n)
    if cleaned_node_id in node_in_gnomad:
        covered_nodes.append(n)

In [ ]:
print(len(covered_nodes), "||", f"{round((len(covered_nodes) / len(node_in_G))*100, 2)}%")

In [ ]:
len(gnomad_mut_df.uniprot_id.unique())

# Secondary Strcture

In [ ]:
y_df['cleaned_rm_node_id']= y_df['node_id'].map(get_node_id_rm_copy)

In [ ]:
str_ss_df = y_df[['node_id',
                  'dssp_sec_struct', # Class
                  'dssp_helix_3_10', 'dssp_helix_alpha', 'dssp_helix_pi', 'dssp_helix_pp', 'dssp_bend', # str
                  'dssp_chirality', 'dssp_sheet', 'dssp_strand', 'dssp_ladder_1','dssp_ladder_2',]]  # str

float_ss_df = y_df[['node_id', 'rel_sasa', 'ss_helix', 'ss_sheet','ss_loop', 'depth', 'hse_up', 'hse_down',
                    'dssp_accessibility', 'dssp_TCO', 'dssp_kappa','dssp_alpha', 'dssp_phi', 'dssp_psi',]]


float_ss_df['cleaned_rm_node_id']= float_ss_df['node_id'].map(get_node_id_rm_copy)
str_ss_df['cleaned_rm_node_id']= str_ss_df['node_id'].map(get_node_id_rm_copy)

In [ ]:
float_cols = ['rel_sasa', 'ss_helix', 'ss_sheet','ss_loop', 'depth', 'hse_up', 'hse_down',
              'dssp_accessibility', 'dssp_TCO', 'dssp_kappa','dssp_alpha', 'dssp_phi', 'dssp_psi',]

In [ ]:
float_ss_df['uniprot'] = float_ss_df['cleaned_rm_node_id'].apply(lambda x: x.split('_')[0])
float_ss_df['res'] = float_ss_df['cleaned_rm_node_id'].apply(lambda x: x.split('_')[2])
float_ss_df['pos'] = float_ss_df['cleaned_rm_node_id'].apply(lambda x: int(x.split('_')[1]))
float_ss_df

In [ ]:
# NaN Augmentation (1) - Origin Residue based Augmentaiton for copy
float_ss_df.set_index('node_id', inplace=True)

for col in float_cols:
    group_means = float_ss_df.groupby('cleaned_rm_node_id')[col].mean()
    is_na = float_ss_df[col].isna()
    float_ss_df.loc[is_na, col] = float_ss_df.loc[is_na, 'cleaned_rm_node_id'].map(group_means)

float_ss_df.reset_index(inplace=True)

In [ ]:
# NaN Augmentation (2) - Neighbor based Augmentaiton
for col in float_cols:
    na_rows = float_ss_df[float_ss_df[col].isna()]
    
    for idx, row in na_rows.iterrows():
        u_id = row['uniprot']
        res_type = row['res']
        current_pos = row['pos']
        mask = (
            (float_ss_df['uniprot'] == u_id) & 
            (float_ss_df['res'] == res_type) & 
            (float_ss_df['pos'] >= current_pos - 20) & 
            (float_ss_df['pos'] <= current_pos + 20)
        )
        
        nearby_mean = float_ss_df.loc[mask, col].mean()
        
        if pd.notna(nearby_mean):
            float_ss_df.at[idx, col] = nearby_mean

In [ ]:
merge_float_df = float_ss_df.groupby('node_id').mean().reset_index()
merge_float_df.info()

# Evolutionary Information

In [ ]:
with open('proc_data/pssm_features.json') as f:
    #  A, R, N, D, C, Q, E, G, H, I, L, K, M, F, P, S, T, W, Y, V
    pssm = json.load(f)

with open('proc_data/hmm_features.json') as f:
    # A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y
    hmm = json.load(f)

In [ ]:
pssm

In [ ]:
hmm